In [17]:
from ifc.config import load_data_config, load_split_config, resolve_project_root
from pathlib import Path
import pandas as pd

data_cfg = load_data_config() 
split_cfg = load_split_config()  
train_path, test_path = data_cfg.resolve_paths()    
print(data_cfg,"\n")

print(f'training path:{train_path}, exists? {train_path.exists()}')
print(f'tets path:{test_path}, exists? {test_path.exists()}')

id_col, time_col, target = data_cfg.id_col, data_cfg.time_col, data_cfg.target_col

print(f"Company_id column: {id_col}\nFiscal_year column: {time_col}\nTarget column: {target}")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)


DataConfig(train_path=WindowsPath('data/processed/train_data.csv'), test_path=WindowsPath('data/processed/test_features.csv'), id_col='company_id', time_col='fiscal_year', target_col='revenue_change') 

training path:C:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\data\processed\train_data.csv, exists? True
tets path:C:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\data\processed\test_features.csv, exists? True
Company_id column: company_id
Fiscal_year column: fiscal_year
Target column: revenue_change
train_df: (11828, 30)
test_df : (5811, 27)


In [18]:
train_df[[time_col,target]].describe().T

,count,mean,std,min,25%,50%,75%,max
fiscal_year,11828.0,2019.49535,1.116482,2018.00,2018.00,2019.00,2020.00,2021.00
revenue_change,8829.0,453.43457,4601.920625,-99.94,-68.59,3.04,238.85,302126.48


In [19]:
test_df[time_col].describe().T

count    5811.000000
mean     2022.498193
std         0.500040
min      2022.000000
25%      2022.000000
50%      2022.000000
75%      2023.000000
max      2023.000000
Name: fiscal_year, dtype: float64

In [20]:
unique_id = train_df[id_col].nunique()
unique_id_percentage = unique_id/len(train_df[id_col])

print(f'unique IDs:{unique_id}')
print(f'unique id percentage:{unique_id_percentage}')

unique IDs:2999
unique id percentage:0.25355089617855936


In [21]:
def missing_summary(df):
    missing_values = df.isnull().sum()
    missing_pct = (missing_values / len(df)) * 100

    missing_df = pd.DataFrame({
        'Missing Count': missing_values,
        'Percentage': missing_pct
    })
    missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
    return missing_df




In [22]:
missing_train = missing_summary(train_df)
print("Missing Values train:")
print(missing_train)

Missing Values train:
                Missing Count  Percentage
revenue_change           2999   25.355090
province                  919    7.769699
roe                        45    0.380453
leverage                   45    0.380453


In [23]:
missing_test = missing_summary(test_df)
print("Missing Values test")
print(missing_test)

Missing Values test
          Missing Count  Percentage
province            451    7.761143
roe                  14    0.240922
leverage             14    0.240922


In [24]:
key = [id_col, time_col]

dup_train = train_df.duplicated(subset=key).sum()
print("dup_train:", dup_train)

dup_test = test_df.duplicated(subset=key).sum()
print("dup_test:", dup_test)

if dup_train > 0:
    display(train_df.loc[train_df.duplicated(subset=key, keep=False), key].sort_values(key).head(20))

if dup_test > 0:
    display(test_df.loc[test_df.duplicated(subset=key, keep=False), key].sort_values(key).head(20))


dup_train: 0
dup_test: 0


In [25]:

df = train_df.copy()

first_year = df.groupby("company_id")["fiscal_year"].min()
df = df.join(first_year.rename("first_year"), on="company_id")

missing_in_first = df.loc[df["revenue_change"].isna() & (df["fiscal_year"] == df["first_year"])].shape[0]
missing_total = df["revenue_change"].isna().sum()

missing_not_first = df.loc[df["revenue_change"].isna() & (df["fiscal_year"] != df["first_year"])].shape[0]

print("missing_total:", missing_total)
print("missing_in_first_year:", missing_in_first)
print("missing_not_first_year (anomalies):", missing_not_first)

if missing_not_first > 0:
    display(df.loc[df["revenue_change"].isna() & (df["fiscal_year"] != df["first_year"]),
                   ["company_id","fiscal_year","first_year","production_value"]].head(20))


missing_total: 2999
missing_in_first_year: 2999
missing_not_first_year (anomalies): 0


In [26]:
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)

only_in_train = sorted(list(train_cols - test_cols))
only_in_test = sorted(list(test_cols - train_cols))

print("Only in train:", only_in_train)
print("Only in test :", only_in_test)


Only in train: ['bankruptcy_next_year', 'financial_health_class', 'revenue_change']
Only in test : []


In [27]:
years = sorted(train_df[time_col].unique())
print("years_in_train_df:", years)

# counts per year
print(train_df[time_col].value_counts().sort_index())


years_in_train_df: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
fiscal_year
2018    2961
2019    2979
2020    2956
2021    2932
Name: count, dtype: int64


# Data split

In [28]:
import sys
from ifc.config import load_data_config, load_split_config

data_cfg = load_data_config()
split_cfg = load_split_config()
train_path, test_path = data_cfg.resolve_paths()

print(sys.executable)
print(train_path, train_path.exists())
print(test_path, test_path.exists())
print(split_cfg)


c:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\.venv\Scripts\python.exe
C:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\data\processed\train_data.csv True
C:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\data\processed\test_features.csv True
SplitConfig(strategy='holdout', time_col='fiscal_year', id_col='company_id', target_col='revenue_change', shuffle=False, history_years=[2018], train_years=[2019, 2020], val_years=[2021])


In [29]:
from ifc.config import load_split_config
from ifc.split import apply_holdout_split

split_cfg = load_split_config()

df_history, df_train, df_val = apply_holdout_split(train_df, split_cfg)

print("history:", df_history.shape, sorted(df_history["fiscal_year"].unique()))
print("train  :", df_train.shape, sorted(df_train["fiscal_year"].unique()))
print("val    :", df_val.shape, sorted(df_val["fiscal_year"].unique()))

print("train target missing:", df_train["revenue_change"].isna().sum())
print("val target missing  :", df_val["revenue_change"].isna().sum())


history: (2961, 30) [np.int64(2018)]
train  : (5897, 30) [np.int64(2019), np.int64(2020)]
val    : (2932, 30) [np.int64(2021)]
train target missing: 0
val target missing  : 0
